# PhySSL-Net: Complete Research Visualisation, Statistical Analysis, and Results Pipeline
## Automated Scientific Results Generation for DTU B.Tech Project Report & Publications

> **Project Title:** Physics- and Structure-Informed Neural Networks with Self-Supervised Learning for Medical Image Classification  
> **Institution:** Delhi Technological University (DTU), Department of Computer Science & Engineering  
> **Project Group:** `DTU/CSE/BTP/GP053` (Academic Year 2026–27)  
> **Project Guide:** Dr. Ravin Ahuja  
> **Authors:** Manish Singh, Shivam, Siddharth Kumar Maurya, Saurav Kumar  

---
### Abstract & Architectural Blueprint:
PhySSL-Net introduces a parameter-efficient (46,318 parameters) architecture incorporating domain-specific physics priors for medical imaging:
1. **Ramanujan-Schoenberg Basis (`AdaptiveSplineGaussianRBF`):** Learnable Gaussian core modulated by truncated power splines over normalized intensity domain $[-1, 1]$.
2. **Dual-Path Band-Pass Residual (`BandPassBlock`):** $7\times 7$ depthwise convolution with absolute modulus $|Z|$ for high-frequency micro-texture and edge isolation.
3. **Level-Set Mean Curvature Attention (`CurvatureAttention`):** Scalar field mean curvature divergence $\text{div}(\nabla u / \|\nabla u\|)$ with normalized Sobel operators.
4. **Global Spatial Token Mixer (`GlobalTokenMixer`):** $7\times 7$ spatial grid MLP cross-position and cross-channel feature communication.
5. **Self-Supervised Masked SimSiam (`PhySSLSimSiam`):** Phase 1 unlabelled representation learning with $4\times 4$ patch masking and stop-gradient cosine loss.

---
### Notebook Organization (15 Sections):
1. Project Overview & Blueprint
2. Kaggle Environment Setup & Dependency Verification
3. Dataset and Results Telemetry Inspection
4. Data Quality & Metric Verification
5. Dataset Analysis & Medical Sample Visualisation
6. Training Dynamics & Convergence Tracking
7. Main Model Benchmark Comparison
8. ROC and Confusion Matrix Analysis
9. Architectural Component Ablation Study
10. Computational Efficiency & Throughput Profiling
11. Multi-Objective Pareto Frontier Analysis
12. Statistical Hypothesis Testing (McNemar & 95% CIs)
13. Complete Suite of Publication Tables (Tables 1–10 in 4 Formats)
14. Final Evidence-Based Research Summary
15. Automated Packaging & Output Download

In [ ]:
# =====================================================================
# 2. KAGGLE ENVIRONMENT SETUP & HARDWARE DETECTION
# =====================================================================
import os
import sys
import glob
import json
import shutil
import math
import time
import platform
import zipfile
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from IPython.display import display, Image, Markdown

# Detect Kaggle environment and configure working paths
IS_KAGGLE = os.path.exists("/kaggle")
BASE_DIR = "/kaggle/working/results" if IS_KAGGLE else "./results"
DATA_DIR = "/kaggle/input/pneumoniamnist" if (IS_KAGGLE and os.path.exists("/kaggle/input/pneumoniamnist")) else "./data"

# Auto-install openpyxl if missing for Excel export
try:
    import openpyxl
except ImportError:
    !pip install -q openpyxl
    import openpyxl

# Publication-grade Matplotlib formatting
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 13,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
    'grid.color': '#E0E0E0',
    'grid.linestyle': '--',
    'grid.linewidth': 0.5,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

import torch
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"[*] Environment: {'Kaggle Kernel' if IS_KAGGLE else 'Local Workstation'}")
print(f"[*] Hardware Accelerator: {GPU_NAME}")
print(f"[*] Base Output Directory: {BASE_DIR}")


In [ ]:
# =====================================================================
# 3. DATASET AND RESULTS TELEMETRY INSPECTION
# =====================================================================
from scripts.generate_research_visualizations import create_directory_structure, run_environment_audit, organize_raw_files

# Create standard results directory structure
dirs = create_directory_structure(BASE_DIR)
print(f"[+] Verified and created {len(dirs)} results subdirectories.")

# Execute environment and results audit
audit_stats = run_environment_audit(BASE_DIR, DATA_DIR)
organize_raw_files(BASE_DIR)

print(f"\n--- Telemetry Audit Summary ---")
print(f"Models Audited: {len(audit_stats['models'])} ({audit_stats['models']})")
print(f"Seeds Audited:  {len(audit_stats['seeds'])} ({audit_stats['seeds']})")
print(f"Training Logs:  {audit_stats['log_count']} logs")
print(f"Eval Records:   {audit_stats['eval_count']} records")
print(f"Checkpoints:    {audit_stats['ckpt_count']} weights")

with open(f"{BASE_DIR}/reports/visualization_data_audit.md") as f:
    display(Markdown(f.read()))


In [ ]:
# =====================================================================
# 4. DATA QUALITY VALIDATION & CONSOLIDATION
# =====================================================================
from scripts.generate_research_visualizations import load_and_consolidate_results

benchmark_df, full_df = load_and_consolidate_results(BASE_DIR)
print(f"[+] Ingested {len(benchmark_df)} benchmark evaluation runs across {benchmark_df['model_name'].nunique()} models.")

# Display consolidated benchmark records
summary_view = benchmark_df[['model_name', 'seed', 'accuracy', 'roc_auc', 'f1', 'recall', 'specificity', 'num_parameters']].copy()
summary_view['accuracy'] = (summary_view['accuracy'] * 100).round(2).map(lambda a: f"{a:.2f}%")
summary_view['roc_auc'] = summary_view['roc_auc'].round(4)
summary_view['f1'] = summary_view['f1'].round(4)
summary_view['recall'] = (summary_view['recall'] * 100).round(2).map(lambda r: f"{r:.2f}%")
summary_view['specificity'] = (summary_view['specificity'] * 100).round(2).map(lambda s: f"{s:.2f}%")

display(summary_view.head(10))


In [ ]:
# =====================================================================
# 5. DATASET VISUALISATION & EXPLORATION
# =====================================================================
from scripts.generate_research_visualizations import generate_dataset_visualizations

dataset_npz = os.path.join(DATA_DIR, "pneumoniamnist.npz")
if not os.path.exists(dataset_npz):
    dataset_npz = "./data/pneumoniamnist.npz"

generate_dataset_visualizations(dataset_npz, BASE_DIR)

print("\n[+] Fig 01: Class Distribution:")
display(Image(filename=f"{BASE_DIR}/figures/dataset/fig_01_dataset_distribution.png"))

print("\n[+] Fig 02: Dataset Partition Splits:")
display(Image(filename=f"{BASE_DIR}/figures/dataset/fig_02_dataset_split_distribution.png"))

print("\n[+] Fig 03: Representative Medical Images:")
display(Image(filename=f"{BASE_DIR}/figures/dataset/fig_03_sample_medical_images.png"))

print("\n[+] Fig 04: Preprocessing Normalization Flow:")
display(Image(filename=f"{BASE_DIR}/figures/dataset/fig_04_preprocessing_comparison.png"))


In [ ]:
# =====================================================================
# 6. TRAINING DYNAMICS & CONVERGENCE TRACKING
# =====================================================================
from scripts.generate_research_visualizations import generate_training_visualizations

generate_training_visualizations(BASE_DIR)

print("\n[+] Fig 05: Multi-Model Training Loss Curves:")
display(Image(filename=f"{BASE_DIR}/figures/training/fig_05_training_loss_curves.png"))

print("\n[+] Fig 06: Multi-Model Validation Loss Tracking:")
display(Image(filename=f"{BASE_DIR}/figures/training/fig_06_validation_loss_curves.png"))

print("\n[+] Fig 07: Validation Accuracy Trajectory:")
display(Image(filename=f"{BASE_DIR}/figures/training/fig_07_validation_accuracy_curves.png"))

print("\n[+] Fig 08: Generalization Gap Dynamics:")
display(Image(filename=f"{BASE_DIR}/figures/training/fig_08_overfitting_analysis.png"))

if os.path.exists(f"{BASE_DIR}/figures/training/fig_09_ssl_pretrain_loss.png"):
    print("\n[+] Fig 09: Masked SimSiam SSL Pretraining Curve:")
    display(Image(filename=f"{BASE_DIR}/figures/training/fig_09_ssl_pretrain_loss.png"))


In [ ]:
# =====================================================================
# 7. MAIN BENCHMARK PERFORMANCE COMPARISON
# =====================================================================
from scripts.generate_research_visualizations import generate_benchmark_visualizations

generate_benchmark_visualizations(benchmark_df, BASE_DIR)

print("\n[+] Fig 10: 5-Seed Aggregate Accuracy Comparison:")
display(Image(filename=f"{BASE_DIR}/figures/benchmark/fig_10_model_accuracy_comparison.png"))

print("\n[+] Fig 12: Grouped Metric Performance Profile:")
display(Image(filename=f"{BASE_DIR}/figures/benchmark/fig_12_metrics_grouped_comparison.png"))

print("\n[+] Fig 14: Per-Seed Stability & Metric Variability:")
display(Image(filename=f"{BASE_DIR}/figures/benchmark/fig_14_seed_variability.png"))

print("\n[+] TABLE 5: Five-Seed Performance Summary (Mean ± SD):")
tab5 = pd.read_csv(f"{BASE_DIR}/tables/benchmark/table_05_five_seed_summary.csv")
display(tab5)


In [ ]:
# =====================================================================
# 8. ROC DISCRIMINATION & CONFUSION MATRIX PROFILES
# =====================================================================
print("\n[+] Fig 11: Receiver Operating Characteristic (ROC) Comparison:")
display(Image(filename=f"{BASE_DIR}/figures/benchmark/fig_11_roc_comparison.png"))

print("\n[+] Fig 13: 5-Model Test Confusion Matrices:")
display(Image(filename=f"{BASE_DIR}/figures/benchmark/fig_13_confusion_matrix_comparison.png"))


In [ ]:
# =====================================================================
# 9. ARCHITECTURAL COMPONENT ABLATION STUDY
# =====================================================================
from scripts.generate_research_visualizations import generate_ablation_visualizations

generate_ablation_visualizations(benchmark_df, BASE_DIR)

print("\n[+] Fig 15: Ablation Metric Performance Comparison:")
display(Image(filename=f"{BASE_DIR}/figures/ablation/fig_15_ablation_comparison.png"))

print("\n[+] Fig 16: Parameter Budget Across Ablations:")
display(Image(filename=f"{BASE_DIR}/figures/ablation/fig_16_ablation_parameters.png"))

print("\n[+] Fig 17: Performance Penalty Induced by Component Removal:")
display(Image(filename=f"{BASE_DIR}/figures/ablation/fig_17_component_removal_impact.png"))

print("\n[+] TABLE 6: Architectural Ablation Study Summary:")
tab6 = pd.read_csv(f"{BASE_DIR}/tables/ablation/table_06_ablation_study.csv")
display(tab6)


In [ ]:
# =====================================================================
# 10. COMPUTATIONAL EFFICIENCY & INFERENCE LATENCY
# =====================================================================
from scripts.generate_research_visualizations import generate_efficiency_visualizations

generate_efficiency_visualizations(BASE_DIR)

print("\n[+] Fig 18: Accuracy vs. Model Parameter Budget:")
display(Image(filename=f"{BASE_DIR}/figures/efficiency/fig_18_accuracy_vs_parameters.png"))

print("\n[+] Fig 19: Accuracy vs. CPU Inference Latency:")
display(Image(filename=f"{BASE_DIR}/figures/efficiency/fig_19_accuracy_vs_latency.png"))

print("\n[+] TABLE 7: Computational Efficiency Profile:")
tab7 = pd.read_csv(f"{BASE_DIR}/tables/efficiency/table_07_computational_efficiency.csv")
display(tab7)


In [ ]:
# =====================================================================
# 11. MULTI-OBJECTIVE PARETO FRONTIER ANALYSIS
# =====================================================================
print("\n[+] Fig 20: Accuracy vs. Parameter Budget Pareto Frontier:")
display(Image(filename=f"{BASE_DIR}/figures/efficiency/fig_20_pareto_frontier.png"))


In [ ]:
# =====================================================================
# 12. STATISTICAL SIGNIFICANCE & HYPOTHESIS TESTING
# =====================================================================
from scripts.generate_research_visualizations import generate_statistical_visualizations

generate_statistical_visualizations(BASE_DIR)

print("\n[+] Fig 21: McNemar Paired Test p-values vs. Baselines:")
display(Image(filename=f"{BASE_DIR}/figures/statistics/fig_21_statistical_significance.png"))

print("\n[+] TABLE 8: Statistical Significance Analysis:")
tab8 = pd.read_csv(f"{BASE_DIR}/tables/statistics/table_08_statistical_significance.csv")
display(tab8)


In [ ]:
# =====================================================================
# 13. COMPLETE SUITE OF PUBLICATION RESEARCH TABLES (TABLES 1-10)
# =====================================================================
from scripts.generate_research_visualizations import export_all_research_tables

export_all_research_tables(BASE_DIR)

print("--> TABLE 1: Dataset Summary")
display(pd.read_csv(f"{BASE_DIR}/tables/dataset/table_01_dataset_summary.csv"))

print("--> TABLE 2: Dataset Split Distribution")
display(pd.read_csv(f"{BASE_DIR}/tables/dataset/table_02_dataset_split_distribution.csv"))

print("--> TABLE 3: Model Architecture and Parameters")
display(pd.read_csv(f"{BASE_DIR}/tables/benchmark/table_03_model_architectures.csv"))

print("--> TABLE 9: Hardware and Experimental Configuration")
display(pd.read_csv(f"{BASE_DIR}/tables/benchmark/table_09_experimental_configuration.csv"))

print("--> TABLE 10: Final Research Summary")
display(pd.read_csv(f"{BASE_DIR}/tables/benchmark/table_10_final_research_summary.csv"))


In [ ]:
# =====================================================================
# 14. FINAL EVIDENCE-BASED RESEARCH SUMMARY
# =====================================================================
from scripts.generate_research_visualizations import generate_reports

generate_reports(BASE_DIR)

with open(f"{BASE_DIR}/reports/results_summary.md") as f:
    display(Markdown(f.read()))

with open(f"{BASE_DIR}/reports/visualization_index.md") as f:
    display(Markdown(f.read()))


In [ ]:
# =====================================================================
# 15. AUTOMATED PACKAGING & OUTPUT DOWNLOAD
# =====================================================================
zip_output_path = "/kaggle/working/physsl_net_research_results.zip" if IS_KAGGLE else "./physsl_net_research_results.zip"

print(f"[*] Packaging all research deliverables into {zip_output_path}...")
with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(BASE_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(BASE_DIR))
            zipf.write(file_path, arcname)

file_size_mb = os.path.getsize(zip_output_path) / (1024 * 1024)
print(f"[✓] Package successfully created: {zip_output_path} ({file_size_mb:.2f} MB)")
print(f"[✓] All figures (PNG 300 DPI, PDF, SVG), tables (CSV, XLSX, MD, LaTeX), and markdown reports are ready for academic report submission!")
